## Airline AI Assistant


In [1]:
# Imports: `gradio` for building the chat UI and `OpenAI` for model client usage

import gradio as gr
from openai import OpenAI

d:\CodeBase\GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Start the Ollama server (shell magic). Run this in a notebook cell or terminal to launch the local model server.

!ollama serve

^C


In [3]:
# Quick health-check: verify the Ollama server is reachable before making requests
import requests
requests.get("http://localhost:11434").content

b'Ollama is running'

In [15]:
# Initialize the Ollama/OpenAI client pointing at the local server

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [31]:
# System prompt: defines assistant behavior, tone, and response constraints

system_prompt = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 2 sentences.
Always be accurate. If you don't know the answer, say so.
Only call tools when the user explicitly asks for information that requires a tool (like price, time, or flight details).
"""

In [ ]:
# Chat function: build message list, stream completions from Ollama, and yield incremental responses to the UI

def chat(user_message, chat_history):
    messages = (
        [{"role": "system", "content": system_prompt}]
        + chat_history
        + [{"role": "user", "content": user_message}]
    )

    with ollama.chat.completions.stream(
        model="ministral-3:3B", messages=messages) as stream:
        response = ""
        for chunk in stream:
            if chunk.type == "content.delta":
                response += chunk.delta or ""
                yield response  # yield only when new content arrives

In [2]:
# Launch the Gradio Chat UI with the streaming `chat` function

gr.ChatInterface(fn=chat, title="Chatbot", description="Airline AI Assistant").launch(show_error=True)

NameError: name 'gr' is not defined

## Tools
- tools are functions that the assistant can call to perform specific tasks. They are defined in the `tools` directory and can be imported into the main assistant code.
- Each tool should have a clear purpose and be designed to handle a specific task related to airline operations, such as booking flights, checking flight status, or providing customer support.
- The assistant can call these tools as needed to assist users with their inquiries and requests.

In [6]:
# Sample flights dataset used by the assistant tools (list of dicts)

flights = [
    {"flight_id": "FL-001", "origin_city": "Delhi", "destination_city": "Mumbai", "time": "08:00", "price": 5500},
    {"flight_id": "FL-002", "origin_city": "Mumbai", "destination_city": "Bangalore", "time": "10:30", "price": 4800},
    {"flight_id": "FL-003", "origin_city": "Delhi", "destination_city": "Kolkata", "time": "12:15", "price": 6200},
    {"flight_id": "FL-004", "origin_city": "Bangalore", "destination_city": "Chennai", "time": "14:45", "price": 3900},
    {"flight_id": "FL-005", "origin_city": "Mumbai", "destination_city": "Delhi", "time": "16:20", "price": 5500},
    {"flight_id": "FL-006", "origin_city": "Hyderabad", "destination_city": "Pune", "time": "09:00", "price": 3200},
    {"flight_id": "FL-007", "origin_city": "Kolkata", "destination_city": "Mumbai", "time": "11:30", "price": 6800},
    {"flight_id": "FL-008", "origin_city": "Chennai", "destination_city": "Delhi", "time": "13:45", "price": 7200},
    {"flight_id": "FL-009", "origin_city": "Pune", "destination_city": "Bangalore", "time": "15:00", "price": 4500},
    {"flight_id": "FL-010", "origin_city": "Hyderabad", "destination_city": "Delhi", "time": "17:30", "price": 5900}
]

In [7]:
# Step 1: Define tool functions that the assistant can call
# - `get_ticket_price`: find a flight by origin/destination and return price
# - `get_departure_time`: find a flight and return departure time
# - `get_flight_details`: find a flight by flight_id and return detailed info

def get_ticket_price(origin, destination):
    for flight in flights:
        if flight["origin_city"].lower() == origin.lower() and flight["destination_city"].lower() == destination.lower():
            return f"The ticket price from {origin} to {destination} is ₹{flight['price']}."
    return f"No flight found from {origin} to {destination}."

def get_departure_time(origin, destination):
    for flight in flights:
        if flight["origin_city"].lower() == origin.lower() and flight["destination_city"].lower() == destination.lower():
            return f"The departure time from {origin} to {destination} is {flight['time']}."
    return f"No flight found from {origin} to {destination}."

def get_flight_details(flight_id):
    for flight in flights:
        if flight["flight_id"].lower() == flight_id.lower():
            return f"Flight {flight_id} details: Origin - {flight['origin_city']}, Destination - {flight['destination_city']}, Time - {flight['time']}, Price - ₹{flight['price']}."
    return f"No flight found with ID {flight_id}."

In [ ]:
# Step 2: Tool metadata used for function-calling (e.g., for an LLM that supports calling typed tools)

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": "Get the ticket price for a flight between two cities. Example: Delhi to Mumbai.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "Origin city"},
                    "destination": {"type": "string", "description": "Destination city"}
                },
                "required": ["origin", "destination"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_departure_time",
            "description": "Get the departure time for a flight between two cities",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "Origin city"},
                    "destination": {"type": "string", "description": "Destination city"}
                },
                "required": ["origin", "destination"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_flight_details",
            "description": "Get detailed information about a specific flight by ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_id": {"type": "string", "description": "Flight ID (e.g., FL-001)"}
                },
                "required": ["flight_id"]
            }
        }
    }
]

In [9]:
# Step 3: Map tool names to the actual Python function objects so the assistant can call them

tool_functions = {
    "get_ticket_price": get_ticket_price,
    "get_departure_time": get_departure_time,
    "get_flight_details": get_flight_details
}

In [28]:
# Step 4: Function to process and execute tool calls from the model response
import json

def execute_tool_call(tool_name, tool_input):
    """
    Execute a tool function based on the tool name and input parameters.
    
    Args:
        tool_name (str): The name of the tool to call
        tool_input (dict or str): Input parameters for the tool
    
    Returns:
        str: The result of the tool execution or error message
    """
    if tool_name in tool_functions:
        try:
            # If tool_input is a string (JSON from model), parse it first
            if isinstance(tool_input, str):
                tool_input = json.loads(tool_input)
                
            # Call the function with unpacked arguments
            result = tool_functions[tool_name](**tool_input)
            return result
        except Exception as e:
            return f"Error executing {tool_name}: {str(e)}"
    else:
        return f"Tool {tool_name} not found."

In [36]:
# Step 5: Enhanced chat function with tool-calling support
# This version sends tool definitions to the model, processes tool calls, and continues the conversation

def chat_with_tools(user_message, chat_history):
    """
    Enhanced chat function that supports tool calling.
    The model can decide to call tools to gather flight information before responding.
    """
    messages = (
        [{"role": "system", "content": system_prompt}]
        + chat_history
        + [{"role": "user", "content": user_message}]
    )

    # Send request with tools available for the model to call
    response = ollama.chat.completions.create(
        model="qwen3.5:4b-q4_K_M",  # Use a tool-capable model
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    # Check if model returned tool calls or regular message
    assistant_message = response.choices[0].message
    
    if assistant_message.tool_calls:
        # Create a list to keep track of the results for this turn
        final_response_parts = []
        
        # Add assistant's tool call intention to history
        messages.append(assistant_message)
        
        # Process each tool call made by the model
        for tool_call in assistant_message.tool_calls:
            tool_name = tool_call.function.name
            tool_input = tool_call.function.arguments
            
            # Execute the tool and get result
            tool_result = execute_tool_call(tool_name, tool_input)
            
            # Append the tool result back to the messages for the next turn
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": str(tool_result)
            })
            final_response_parts.append(f"[System: Checked {tool_name}]")

        # Get a final conversational response from the model based on the tool results
        second_response = ollama.chat.completions.create(
            model="llama3.2",
            messages=messages
        )
        yield second_response.choices[0].message.content
    else:
        # Model returned a direct response (e.g., generic greeting) without tool calls
        yield assistant_message.content

In [37]:
# Launch the Gradio Chat UI with the streaming `chat_with_tools` function

gr.ChatInterface(fn=chat_with_tools, title="Chatbot", description="Airline AI Assistant").launch(show_error=True)

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
